In [1]:
# pip install mygene

In [1]:
import pandas as pd
import numpy as np
import requests
import time
import mygene

In [2]:
df = pd.read_csv("datosGene4PD/t_common_variant.txt", sep = "\t", index_col = False)

In [3]:
df.head()

,Chr,gene_symbol,SNPs_symbol,SNP_position,effect_allele,alternate_allele,joint_phase_P,joint_phase_OR,joint_phase_OR_CI,pubMed_ID,Unnamed: 10
0,6,GPR126,rs757765789,142758601,T,G,1.42E-06,1.06,1.016-1.098,28256260,NaN
1,1,SYT11,rs202015799,155839054,C,T,4.70E-09,-,-,24842889,NaN
2,12,SLC2A13,rs1994090,40428561,G,T,3.20E-54,12.05,8.35-17.41,24842889,NaN
3,12,SLC2A13,rs2708453,40478652,G,T,3.62E-54,12.05,8.35-17.41,24842889,NaN
4,12,SLC2A13,rs4768212,40474147,C,T,3.62E-54,12.05,8.35-17.41,24842889,NaN


In [4]:
df_genes = df["gene_symbol"]

In [5]:
df_genes

0        GPR126
1         SYT11
2       SLC2A13
3       SLC2A13
4       SLC2A13
         ...   
1053     INPP5F
1054       RIT2
1055       GCH1
1056    SIPA1L2
1057    TMPRSS9
Name: gene_symbol, Length: 1058, dtype: object

In [6]:
df_genes_t = df_genes.dropna()

In [7]:
df_genes_t = df_genes_t.reset_index(drop=False)["gene_symbol"]

In [8]:
df_genes_t

0        GPR126
1         SYT11
2       SLC2A13
3       SLC2A13
4       SLC2A13
         ...   
1051     INPP5F
1052       RIT2
1053       GCH1
1054    SIPA1L2
1055    TMPRSS9
Name: gene_symbol, Length: 1056, dtype: object

In [9]:
df_genes_t.head(20)

0              GPR126
1               SYT11
2             SLC2A13
3             SLC2A13
4             SLC2A13
5             SLC2A15
6     LINC02471;LRRK2
7               LRRK2
8               LRRK2
9         GPRIN3;SNCA
10               MAPT
11    LINC02210-CRHR1
12        GPRIN3;SNCA
13    LINC02210-CRHR1
14           SERPINA1
15           SERPINA1
16        S1PR1;OLFM3
17               MAPT
18               MAPT
19             SPPL2C
Name: gene_symbol, dtype: object

In [10]:
def extrae_gene_symbols(dataframe):

    df_corregido = df.drop('Unnamed: 10', axis = 1)
    df_corregido = df_corregido.dropna(subset = ['gene_symbol'])
    df_corregido = df_corregido.reset_index(drop=False)
    df_corregido = df_corregido.drop('index', axis = 1)
    df_genes = df_corregido["gene_symbol"]
    
    lista_symbols = []
    
    for i, gene in enumerate(df_genes):
        
        gene = gene.replace(",", ";")
    
        if "dist" in gene:
            continue

        elif ";" in gene:

            separacion1 = gene.split(";")

            for gen in separacion1:
                lista_symbols.append(gen)

        else:
            lista_symbols.append(gene)
            
    return df_corregido, lista_symbols

In [11]:
df_corregido, lista_symbols = extrae_gene_symbols(df)

In [12]:
# for gen in lista_symbols:
#     print(gen)

In [12]:
mg = mygene.MyGeneInfo()

In [13]:
gen = ["GPR126"]
resultado_prueba = mg.querymany(gen, scopes = "symbol,alias", fields = "genomic_pos_hg19", species = "human")

In [14]:
print(resultado_prueba)

[{'query': 'GPR126', '_id': '57211', '_score': 18.994669, 'genomic_pos_hg19': {'chr': '6', 'end': 142767403, 'start': 142622991, 'strand': 1}}]


In [15]:
lista_unicos = []
for symbol in lista_symbols:
    if symbol not in lista_unicos:
        lista_unicos.append(symbol)

In [16]:
resultados_coords = mg.querymany(lista_unicos, scopes = "symbol,alias", fields = "genomic_pos_hg19", species = "human")

43 input query terms found dup hits:	[('TBC1D3P2', 2), ('CAST', 4), ('DHFRP3', 2), ('HLA-DQB1', 2), ('BST1', 2), ('CASC6', 2), ('PWRN4', 
48 input query terms found no hit:	['SLC2A15', 'NONE', 'LOC440311', '43160', 'LOC100133091', 'LOC101928978', 'MIR7641-2', 'LOC201175', 


In [24]:
# print(resultados_coords)

In [17]:
coords_genes = []
no_encontrados = []

for resultado in resultados_coords:
    
    gen = resultado.get("query")

    if resultado.get("notfound"):
        no_encontrados.append(gen)
        continue

    posicion = resultado.get("genomic_pos_hg19")
    
    if isinstance(posicion, list):
        posicion = posicion[0]

    if posicion:
        coords_genes.append({"gene_symbol": gen, "chr": str(posicion.get("chr")), "inicio": posicion.get("start"), "fin": posicion.get("end"), "cadena": posicion.get("strand")})



In [36]:
# no_encontrados

In [38]:
# coords_genes

In [18]:
df_coords = pd.DataFrame(coords_genes)

In [19]:
df_coords

,gene_symbol,chr,inicio,fin,cadena
0,GPR126,6,142622991,142767403,1
1,SYT11,1,155829300,155854990,1
2,SLC2A13,12,40148823,40499891,-1
3,LRRK2,12,40590546,40763087,1
4,GPRIN3,4,90157537,90229161,-1
...,...,...,...,...,...
594,DNAH17,17,76419778,76573476,-1
595,ASXL3,18,31158579,31331156,1
596,MEX3C,18,48700920,48744674,-1
597,CRLS1,20,5986736,6020699,1


In [61]:
# df['Unnamed: 10'].isna().all()

True

In [20]:
df_corregido

,Chr,gene_symbol,SNPs_symbol,SNP_position,effect_allele,alternate_allele,joint_phase_P,joint_phase_OR,joint_phase_OR_CI,pubMed_ID
0,6,GPR126,rs757765789,142758601,T,G,1.42E-06,1.06,1.016-1.098,28256260
1,1,SYT11,rs202015799,155839054,C,T,4.70E-09,-,-,24842889
2,12,SLC2A13,rs1994090,40428561,G,T,3.20E-54,12.05,8.35-17.41,24842889
3,12,SLC2A13,rs2708453,40478652,G,T,3.62E-54,12.05,8.35-17.41,24842889
4,12,SLC2A13,rs4768212,40474147,C,T,3.62E-54,12.05,8.35-17.41,24842889
...,...,...,...,...,...,...,...,...,...,...
1051,rs117896735,INPP5F,13,U,13,U,1.21E-11,1.77,-,29700661
1052,rs12456492,RIT2,21,U,21,U,2.15E-11,1.1,-,29700661
1053,rs7155501,GCH1,15,U,15,U,1.25E-10,1.12,-,29700661
1054,rs10797576,SIPA1L2,21,U,21,U,1.76E-10,1.13,-,29700661


In [21]:
df_SNPs = df_corregido[["Chr", "gene_symbol", "SNP_position", "effect_allele", "alternate_allele"]]

In [22]:
df_SNPs

,Chr,gene_symbol,SNP_position,effect_allele,alternate_allele
0,6,GPR126,142758601,T,G
1,1,SYT11,155839054,C,T
2,12,SLC2A13,40428561,G,T
3,12,SLC2A13,40478652,G,T
4,12,SLC2A13,40474147,C,T
...,...,...,...,...,...
1051,rs117896735,INPP5F,U,13,U
1052,rs12456492,RIT2,U,21,U
1053,rs7155501,GCH1,U,15,U
1054,rs10797576,SIPA1L2,U,21,U


In [23]:
df_SNPs = df_SNPs[~df_SNPs['Chr'].str.contains('rs', na = False)]

In [24]:
df_SNPs

,Chr,gene_symbol,SNP_position,effect_allele,alternate_allele
0,6,GPR126,142758601,T,G
1,1,SYT11,155839054,C,T
2,12,SLC2A13,40428561,G,T
3,12,SLC2A13,40478652,G,T
4,12,SLC2A13,40474147,C,T
...,...,...,...,...,...
1039,17,DNAH17,76425480,A,T
1040,18,ASXL3,31304318,T,G
1041,18,MEX3C,48683589,T,G
1042,20,CRLS1,6006041,T,C


In [32]:
for i in range(len(df_SNPs)):
    if 
    loc_genes = {df_SNPs.iloc[i]['gene_symbol']: {'Chr': df_SNPs.iloc[i]['Chr'], 'SNP_pos': df_SNPs.iloc[i]['SNP_position']}}
    break

In [33]:
loc_genes

{'GPR126': {'Chr': '6', 'SNP_pos': '142758601'}}

In [44]:
dicc = {}
for gen in lista_unicos:
    dicc.update({gen: {'Chr':0, 'Inicio':0, 'SNPs': [], 'Fin':0, 'Cadena':0}})

In [57]:
for i in range(len(df_SNPs)):
    
    gen = df_SNPs.iloc[i]['gene_symbol']
    
    if lista_symbols.count(gen) == 1:
        
        dicc[gen]['Chr'] = df_SNPs.iloc[i]['Chr']
        dicc[gen]['SNPs'] = df_SNPs.iloc[i]['SNP_position']
        
    elif lista_symbols.count(gen) > 1:
        if len(dicc[gen]['SNPs']) == 0:
            
            dicc[gen]['Chr'] = df_SNPs.iloc[i]['Chr']
            dicc[gen]['SNPs'].append(df_SNPs.iloc[i]['SNP_position'])

        else:
            dicc[gen]['SNPs'].append(df_SNPs.iloc[i]['SNP_position'])
    else:
        continue

In [61]:
lista_symbols.count('SLC2A13')

14

In [60]:
dicc

{'GPR126': {'Chr': '6',
  'Inicio': 0,
  'SNPs': '142758601',
  'Fin': 0,
  'Cadena': 0},
 'SYT11': {'Chr': 1,
  'Inicio': 0,
  'SNPs': [155839054, 154105678, '155839054', '154105678'],
  'Fin': 0,
  'Cadena': 0},
 'SLC2A13': {'Chr': 12,
  'Inicio': 0,
  'SNPs': [40428561,
   40478652,
   40474147,
   40428561,
   40458384,
   40465786,
   40428561,
   40465942,
   40478652,
   40474147,
   '40428561',
   '40478652',
   '40474147',
   '40428561',
   '40458384',
   '40465786',
   '40428561',
   '40465942',
   '40478652',
   '40474147',
   '40442892',
   '40385118',
   '40493416'],
  'Fin': 0,
  'Cadena': 0},
 'SLC2A15': {'Chr': '14',
  'Inicio': 0,
  'SNPs': '40465942',
  'Fin': 0,
  'Cadena': 0},
 'LINC02471': {'Chr': 0, 'Inicio': 0, 'SNPs': [], 'Fin': 0, 'Cadena': 0},
 'LRRK2': {'Chr': 12,
  'Inicio': 0,
  'SNPs': [40620808,
   40734202,
   40734202,
   40757328,
   38907075,
   40614434,
   40653510,
   38907075,
   40580440,
   40632099,
   38917875,
   38907082,
   '40620808',
   '